<a href="https://www.kaggle.com/code/avikdas567/vesuvius-challenge-surface-detection?scriptVersionId=300410980" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import tifffile as tiff
from scipy.ndimage import gaussian_filter, label

INPUT_DIR = "/kaggle/input/vesuvius-challenge-surface-detection"
TEST_IMG_DIR = os.path.join(INPUT_DIR, "test_images")
OUTPUT_DIR = "/kaggle/working/preds"
ZIP_PATH = "/kaggle/working/submission.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load test metadata

test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
print(f"Found {len(test_df)} test volumes")

# LZW-safe TIFF reader

def read_tiff_stack_pil(path):
    imgs = []
    with Image.open(path) as img:
        i = 0
        while True:
            try:
                img.seek(i)
                imgs.append(np.array(img))
                i += 1
            except EOFError:
                break
    return np.stack(imgs, axis=0)

# Surface detection

def improved_surface_detection(volume):
    """
    Topology-safe surface extraction using:
    - Z-gradient ridge detection
    - 3D smoothing
    - connected component cleanup
    """
    Z, Y, X = volume.shape

    # normalize
    v = volume.astype(np.float32)
    v = (v - v.min()) / (v.max() - v.min() + 1e-6)

    # Z-gradient magnitude
    grad = np.abs(np.diff(v, axis=0, prepend=v[:1]))

    # smooth gradients
    grad = gaussian_filter(grad, sigma=(1.2, 1.0, 1.0))

    # per-column normalization
    grad /= (grad.max(axis=0, keepdims=True) + 1e-6)

    # soft surface probability
    prob = grad ** 1.5

    # global threshold
    thresh = np.quantile(prob, 0.88)
    mask = (prob > thresh).astype(np.uint8)

    # thicken surface conservatively in Z
    thick = np.zeros_like(mask)
    for dz in (-1, 0, 1):
        z0 = max(0, dz)
        z1 = Z + min(0, dz)
        thick[z0:z1] |= mask[z0 - dz:z1 - dz]
    mask = thick

    # keep largest connected component only (26-connectivity)
    labeled, ncc = label(mask)
    if ncc > 1:
        sizes = np.bincount(labeled.ravel())
        sizes[0] = 0
        keep = sizes.argmax()
        mask = (labeled == keep).astype(np.uint8)

    return mask

# Run inference

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    vid = row["id"]
    img_path = os.path.join(TEST_IMG_DIR, f"{vid}.tif")

    volume = read_tiff_stack_pil(img_path)
    pred = improved_surface_detection(volume)

    tiff.imwrite(
        os.path.join(OUTPUT_DIR, f"{vid}.tif"),
        pred.astype(np.uint8)
    )

# Submission

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, fname), fname)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Submission created!")
    print("Files:", len(z.namelist()))
    print("Example:", z.namelist()[:5])

print("READY TO SUBMIT:", ZIP_PATH)

Found 1 test volumes


100%|██████████| 1/1 [00:04<00:00,  4.23s/it]


Submission created!
Files: 1
Example: ['1407735.tif']
READY TO SUBMIT: /kaggle/working/submission.zip
